# Mandarin (EN→ZH) Fine-Tuning Pipeline — Sprint 51, Task A

Reproducible pipeline: data validation/versioning/hashing → hardware smoke test →
LoRA/QLoRA fine-tuning → checkpoint load + generation → `candidate_manifest.json`.

**Cantonese is intentionally excluded from this pipeline.**

Run cells top to bottom. Recommended Colab runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU).


## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
# --- Option A: clone your GitHub repo (recommended for the "exact GitHub link" deliverable) ---
# !git clone https://github.com/<your-org>/<your-repo>.git
# %cd <your-repo>

# --- Option B: upload the provided mandarin_pipeline.zip directly to this Colab session ---
import os, zipfile

ZIP_NAME = "mandarin_pipeline.zip"

if not os.path.exists(ZIP_NAME) and not os.path.exists("mandarin_pipeline"):
    print(f"'{ZIP_NAME}' not found in {os.getcwd()} — opening the upload dialog.")
    print("Select mandarin_pipeline.zip from your computer.")
    from google.colab import files
    uploaded = files.upload()  # blocks until you pick a file
    # in case the uploaded filename differs, pick the first .zip we got
    zips = [f for f in uploaded if f.endswith(".zip")]
    if zips:
        ZIP_NAME = zips[0]

if os.path.exists("mandarin_pipeline"):
    print("mandarin_pipeline/ already present, skipping extraction.")
elif os.path.exists(ZIP_NAME):
    with zipfile.ZipFile(ZIP_NAME) as z:
        z.extractall(".")
    print(f"Extracted {ZIP_NAME}.")
else:
    raise FileNotFoundError(
        f"Still no '{ZIP_NAME}' in {os.getcwd()} after the upload prompt. "
        "Either re-run this cell and upload the file when prompted, or use Option A (git clone) instead."
    )

assert os.path.exists("mandarin_pipeline"), "Extraction did not produce a mandarin_pipeline/ folder — check the zip contents."
%cd mandarin_pipeline
!ls

In [ ]:
!pip install -q -r requirements-cuda.txt
# Colab preinstalls an old torchao that is incompatible with newer peft's
# LoRA dispatch check (raises ImportError even though we don't use torchao
# at all for plain LoRA). Removing it makes peft correctly treat it as absent.
!pip uninstall -y torchao -q

## Upload the gold dataset

This pipeline uses the Sprint 50 judge-validation gold export
(`gold_en_cmn.jsonl`, English→Mandarin direction with human-verified
reference translations) as its dataset. Upload it into the
`data/raw/` folder here.


In [ ]:
import os
os.makedirs("data/raw", exist_ok=True)

GOLD_FILE = "data/raw/gold_en_cmn.jsonl"
if not os.path.exists(GOLD_FILE):
    print("gold_en_cmn.jsonl not found in data/raw/ — opening the upload dialog.")
    print("Select gold_en_cmn.jsonl from your computer (gold_cmn_en.jsonl is not needed here).")
    from google.colab import files
    uploaded = files.upload()
    for fname, content in uploaded.items():
        target = os.path.join("data/raw", fname)
        with open(target, "wb") as f:
            f.write(content)
    assert os.path.exists(GOLD_FILE), f"Expected {GOLD_FILE} after upload — check the uploaded filename."
else:
    print(f"Found existing {GOLD_FILE}, skipping upload.")

import json
n = sum(1 for _ in open(GOLD_FILE, encoding="utf-8"))
print(f"{GOLD_FILE}: {n} lines")

## 2. Configure the run

Edit overrides below as needed. Key things to check:
- `run.run_name` — versions this candidate's output folder
- `run.seed` — reproducibility
- `model.base_model` — default is `facebook/nllb-200-distilled-600M` (fits comfortably on a T4)

Note: the default `data.source_type` is `gold_export`, pointed at the
`gold_en_cmn.jsonl` you just uploaded (~40 human-verified examples from the
Sprint 50 judge-validation task). This is a small, high-quality dataset —
fine for producing and validating a candidate checkpoint end-to-end, but
not a substitute for a large corpus in a real production fine-tune. Switch
`data.source_type` to `hf_hub` (see `config/default_config.yaml`) to pull a
much larger EN-ZH corpus (e.g. `Helsinki-NLP/opus-100`) instead.


In [ ]:
RUN_NAME = "en-zh-candidate-colab-01"
SEED = 42
DRY_RUN = "true"   # set to "false" once you've confirmed the dry run works end to end

overrides = [
    f"run.run_name={RUN_NAME}",
    f"run.seed={SEED}",
    f"run.dry_run={DRY_RUN}",
]
print(overrides)

## 3. Dry run first (validates the full pipeline wiring cheaply)

This uses a tiny subset of data and 1–2 training steps just to prove:
data prep → hardware smoke test → train → save checkpoint → reload checkpoint → generate → manifest
all work end-to-end **before** committing a full run to it.


In [ ]:
import subprocess, sys

cmd = [sys.executable, "run_pipeline.py", "--config", "config/default_config.yaml"]
for o in overrides:
    cmd += ["--set", o]

print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("----- STDERR -----")
    print(result.stderr)
print("Return code:", result.returncode)

## 4. Full run

Once the dry run above completes without errors, switch `DRY_RUN` to `"false"` and re-run.
This is the run that must "complete on Kaggle, Colab or local resources" per the acceptance criteria.


In [ ]:
DRY_RUN = "false"
overrides = [
    f"run.run_name={RUN_NAME}",
    f"run.seed={SEED}",
    f"run.dry_run={DRY_RUN}",
]

cmd = [sys.executable, "run_pipeline.py", "--config", "config/default_config.yaml"]
for o in overrides:
    cmd += ["--set", o]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("----- STDERR -----")
    print(result.stderr)
print("Return code:", result.returncode)

### Resuming a full run

If a Colab session disconnects mid-training, just re-run the same cell above with the same
`run.run_name` — `training.resume_from_checkpoint: "auto"` in the config will pick up the
latest checkpoint automatically. You can also skip already-completed stages, e.g.:

```
!python run_pipeline.py --config config/default_config.yaml \
    --set run.run_name=en-zh-candidate-colab-01 --set run.dry_run=false \
    --skip data_prep hardware_check
```


## 5. Inspect the hardware smoke-test decision

In [ ]:
import json
report_path = f"outputs/{RUN_NAME}/hardware_report.json"
print(json.dumps(json.load(open(report_path)), indent=2))

## 6. Confirm the checkpoint loads and generates a Mandarin translation

In [ ]:
eval_path = f"outputs/{RUN_NAME}/eval_report.json"
eval_report = json.load(open(eval_path, encoding="utf-8"))
print("Checkpoint loaded and generated OK:", eval_report["checkpoint_loaded_and_generated_ok"])
print(f"{eval_report['metric']} score:", eval_report["score"])
print()
for s in eval_report["samples"][:3]:
    print("EN: ", s["source_en"])
    print("REF:", s["reference_zh"])
    print("GEN:", s["candidate_zh"])
    print()

## 7. View the candidate manifest

In [ ]:
manifest_path = f"outputs/{RUN_NAME}/candidate_manifest.json"
manifest = json.load(open(manifest_path, encoding="utf-8"))
print("candidate_id:", manifest["candidate_id"])
print("data_version:", manifest["data"]["data_version"])
print("method used:", manifest["training"]["method"])
print("checkpoint path:", manifest["checkpoint"]["path"])
print()
print(json.dumps({k: v for k, v in manifest.items() if k not in ("config",)}, indent=2, ensure_ascii=False)[:3000])

## 8. Package outputs for submission

Zips config, manifests, logs, report, and the checkpoint together so you can download
them or push to Drive / GitHub Releases (checkpoints are usually too large for a normal git push).


In [ ]:
import shutil
zip_path = shutil.make_archive(f"{RUN_NAME}_submission", "zip", f"outputs/{RUN_NAME}")
print("Packaged:", zip_path)

from google.colab import files
files.download(zip_path)

## 9. (Optional) Push checkpoint + manifest to Hugging Face Hub or Google Drive

Large checkpoint files should not go into git directly. Two common options:

```python
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
shutil.copytree(f"outputs/{RUN_NAME}", f"/content/drive/MyDrive/{RUN_NAME}", dirs_exist_ok=True)
```

```python
# Option B: Hugging Face Hub (requires `huggingface_hub` + a token with write access)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="<your-username>/en-zh-candidate-01", exist_ok=True)
api.upload_folder(folder_path=f"outputs/{RUN_NAME}/final_checkpoint",
                   repo_id="<your-username>/en-zh-candidate-01")
```

Record whichever download/hub path you use as the **"loadable candidate or download path"**
deliverable, alongside the exact GitHub commit link for the code.
